In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

from transformers import AutoTokenizer

import lmppl
import json
import torch
import gc
from collections import defaultdict

from pppl import backends


In [ ]:
paraclite = pd.read_csv("../data/paraclite.csv")

In [ ]:
paraclite = paraclite.sort_values(['doc_name', 'seg_id'])

In [ ]:
LANGUAGE = 'sv'
MODEL_LIST = [
    # 'DTAI-KULeuven/robbert-2023-dutch-base',
    # 'CLTL/MedRoBERTa.nl',
    # 'UMCU/CardioBERTa.nl_clinical',
    'DT4H/CardioBERTa.sv',
    'KB/bert-base-swedish-cased',
]

In [ ]:
paraclite_agg = paraclite.groupby(["doc_name", "src_lang"])\
                    .apply(lambda x: x[LANGUAGE].str.cat(sep="\n"), include_groups=False)\
                    .reset_index()
paraclite_agg.columns = ['doc_name', 'src_lang', 'text']

In [ ]:
# def count_tokens(text, tokenizer_name='DTAI-KULeuven/robbert-2023-dutch-base'):
#     tokenizer = AutoTokenizer.from_pretrained(tokenizer_name)
#     tokens = tokenizer.tokenize(text)
#     return len(tokens)

In [ ]:
# for model in MODEL_LIST:
#     paraclite_agg[f'n_tokens_{model}'] = paraclite_agg['text'].apply(lambda x: count_tokens(x, 
#                                             tokenizer_name=model))

In [ ]:
ppl_dict_ppl = defaultdict(dict)
ppl_dict_minicons = defaultdict(dict)

In [ ]:
def _free(scorer):
    """Best-effort GPU release for any of our backends."""
    # PPLBackend stores `.model`; LmpplBackend wraps lmppl's scorer at
    # `.scorer.model`; MiniconsBackend wraps minicons' scorer at
    # `._scorer.model`. Move whichever exists off-device, then drop refs.
    for attr_chain in (("model",), ("scorer", "model"), ("_scorer", "model")):
        obj = scorer
        for a in attr_chain:
            obj = getattr(obj, a, None)
            if obj is None:
                break
        if obj is not None and hasattr(obj, "to"):
            try:
                obj.to("cpu")
            except Exception:
                pass
    del scorer
    gc.collect()
    torch.cuda.empty_cache()


for _model_name in MODEL_LIST:
    for _src_lang in paraclite_agg.src_lang.unique():
        print(f"Calculating perplexities for {_model_name} and src lang: {_src_lang}")
        texts = list(paraclite_agg.query(f'src_lang=="{_src_lang}"').text.values)

        # print("LMPPL...")
        # #scorer = backends.LmpplStridedBackend(_model_name, windows_size=508, stride=256, batch_size=8, show_progress=True)
        #scorer = backends.LmpplBackend(_model_name, max_length=508, batch_size=4)

        #ppl_dict_lmppl[_model_name] = scorer.get_perplexity(texts[:5])
        #_free(scorer)

        print("Minicons...")
        scorer = backends.MiniconsStridedBackend(
            _model_name, device="cuda", windows_size=508, stride=256, model_batch_size=8, 
            show_progress=False, pll_method='within_word_l2r' # see below
        )
        ppl_dict_minicons[_model_name][_src_lang] = scorer.get_perplexity(texts)
        _free(scorer)

        print("DT4H full document PPL...")
        scorer = backends.PPLBackend(
            _model_name, windows_size=508, stride=256, batch_size=8, device="cuda"
        )
        ppl_dict_ppl[_model_name][_src_lang]= scorer.get_perplexity(texts)
        _free(scorer)

In [ ]:
# # write to json
import json
with open(f'../output/ppl_paraclite_{LANGUAGE}.json', 'w') as f:
     json.dump(ppl_dict_ppl, f)

with open(f'../output/minicons_paraclite_{LANGUAGE}.json', 'w') as f:
     json.dump(ppl_dict_minicons, f)


In [ ]:
with open(f'../output/ppl_paraclite_{LANGUAGE}.json', 'r') as f:
    ppl_dict_ppl = json.loads(f)

with open(f'../output/minicons_paraclite_{LANGUAGE}.json', 'r') as f:
    ppl_dict_minicons = json.loads(f)

TypeError: loads() takes 1 positional argument but 2 were given

In [ ]:
ppl_df = pd.DataFrame()
for model_name, source_dict in ppl_dict_ppl.items():
    for source_name, values in source_dict.items():        
        if model_name in MODEL_LIST:
            _ppl_df = pd.DataFrame()
            _ppl_df['model_name'] =[model_name]*len(values)
            _ppl_df['perplexities'] = values 
            _ppl_df['source'] = [source_name]*len(values)

            ppl_df = pd.concat([ppl_df, _ppl_df], axis=0)


ppl_minicons_df = pd.DataFrame()
for model_name, source_dict in ppl_dict_minicons.items():
    for source_name, values in source_dict.items():        
        if model_name in MODEL_LIST:
            _ppl_df = pd.DataFrame()
            _ppl_df['model_name'] =[model_name]*len(values)
            _ppl_df['perplexities'] = values 
            _ppl_df['source'] = [source_name]*len(values)

            ppl_minicons_df = pd.concat([ppl_minicons_df, _ppl_df], axis=0)


In [ ]:
ppl_df.groupby('model_name').perplexities.aggregate(mean='mean', median='median', geom = lambda x: np.exp(np.mean(np.log(x))))\
.round(2)

In [ ]:
ppl_minicons_df.groupby('model_name').perplexities.aggregate(mean='mean', median='median', geom = lambda x: np.exp(np.mean(np.log(x))))\
.round(2)

In [ ]:
result = ppl_df.groupby(['model_name', 'source']).perplexities.aggregate(mean='mean', median='median', geom = lambda x: np.exp(np.mean(np.log(x))))\
.round(2)

latex = result.to_latex(
    float_format="%.2f",
    multirow=True,
    caption="Perplexity statistics by model and source",
    label="tab:perplexity"
)

print(latex)

In [ ]:
# ppl_minicons_df.groupby(['model_name', 'source']).perplexities.aggregate(mean='mean', median='median', geom = lambda x: np.exp(np.mean(np.log(x))))\
# .round(2)

In [ ]:
sns.boxplot(data=ppl_df, orient='h')
plt.title('Perplexity Distribution on ParaClite Dataset (Dutch Documents)')
plt.xlabel('Perplexity per document')

# Get current y-tick positions and labels
ax = plt.gca()
y_ticks = ax.get_yticks()
y_labels = [label.get_text() for label in ax.get_yticklabels()]

# Prepare new labels and font properties
new_labels = []
fontdicts = []
for text in y_labels:
    if '.nl' in text or 'RobBERT' in text:
        fontdicts.append({'weight': 'bold', 'color': 'green'})
    elif 'Cardio' in text:
        fontdicts.append({'style': 'italic', 'color': 'red'})
    elif 'multi' in text:
        fontdicts.append({'color': 'blue'})
    else:
        fontdicts.append({'color': 'red'})
    new_labels.append(text)

# Set ticks and labels with font properties
ax.set_yticks(y_ticks)
for i, (label, fontdict) in enumerate(zip(new_labels, fontdicts)):
    ax.get_yticklabels()[i].set_fontweight(fontdict.get('weight', 'normal'))
    ax.get_yticklabels()[i].set_fontstyle(fontdict.get('style', 'normal'))
    if 'color' in fontdict:
        ax.get_yticklabels()[i].set_color(fontdict['color'])
    ax.get_yticklabels()[i].set_text(label)
ax.set_yticklabels(new_labels)

# Annotate median and mean perplexities
for i, col in enumerate(ppl_df.columns):
    median = np.median(ppl_df[col].dropna())
    mean = np.mean(ppl_df[col].dropna())
    xpos = ppl_df[col].max()+25  # Place annotation just right of the max value
    ypos = y_ticks[i]
    ax.text(xpos, ypos, f"{median:.1f}", 
            va='center', ha='left',
            fontdict={'weight': 'bold', 'color': 'black'},
            fontsize=10, color='black')


plt.tight_layout()
plt.savefig('../output/ppl_paraclite_distribution.png', dpi=300)


# Perplexity versus MLM proba

In [ ]:
from pppl import ppl_alt

In [ ]:
probs = [0.05, 0.10, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5]
model_names = mod_run_dict.keys()
ppl_res = []

for _model_name in model_names:
    for _prob in tqdm(probs):
        summary, token_results = ppl_alt.process_text(text=paraclite_agg.text.values[0], 
                            model_name=_model_name,
                            model_type='encoder',
                            mask_probability=_prob,
                            min_samples=10)
        
        token_results = [t for t in token_results if np.isinf(t[1]) == np.False_]
        
        ppl_res_dict = {
            'model_name': _model_name,
            'mask_probability': _prob,
            'summary': summary,
            'token_results': token_results
        }
        ppl_res.append(ppl_res_dict)
    

In [ ]:
res_final_df = pd.DataFrame([{
  'model_name': d['model_name'],
  'mask_probability': d['mask_probability'], 
  'median_ppl': np.median([t[1] for t in d['token_results']]),
  'mean_ppl': np.mean([t[1] for t in d['token_results']])} for d in ppl_res])

In [ ]:
ax = plt.figure(figsize=(14,6))
sns.barplot(data=res_final_df, 
            x='mask_probability', 
            y='median_ppl', 
            hue='model_name')
plt.title('Median Perplexity vs Mask Probability')
plt.xlabel('Mask Probability')
plt.ylabel('Median Perplexity')
plt.semilogy()

plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0.)
plt.tight_layout()